# 📊 Indice di Attrattività Comunale della Sardegna

## Obiettivo

Questo notebook calcola un **indice composito di attrattività** per i comuni sardi, basato su 5 pilastri tematici derivati dai dati OpenStreetMap (OSM):

| Pilastro | Peso | Descrizione |
|----------|------|-------------|
| **Turismo** | 30% | Monumenti, musei, chiese, punti di interesse |
| **Natura** | 25% | Parchi, riserve naturali, spiagge, sentieri |
| **Ristorazione** | 20% | Ristoranti, bar, negozi, mercati |
| **Servizi** | 15% | Ospedali, farmacie, bancomat, uffici postali |
| **Infrastrutture** | 10% | Stazioni, fermate bus, parcheggi, stazioni di servizio |

## Fonti Dati

- **osm_per_comune.csv**: conteggi POI per comune (prodotto dal notebook 01)
- **comuni_sardegna.geojson**: confini comunali ISTAT

## Formula

1. **Densità per km²**: $D_{pilastro} = \frac{count_{pilastro}}{area\_km2}$
2. **Normalizzazione Min-Max** (0-100): $N_{pilastro} = \frac{D - D_{min}}{D_{max} - D_{min}} \times 100$
3. **Indice ponderato**: $I = \sum_{pilastro} (N_{pilastro} \times peso_{pilastro})$
4. **Classificazione**: quintili (classe 1-5, dove 5 = massima attrattività)

## Limiti Metodologici

- I dati OSM dipendono dalla comunità di contributori; alcune aree potrebbero essere sottocampionate
- I pesi sono arbitrari e riflettono un'ipotesi di priorità per il turismo
- L'area comunale include aree marine interne (laghi, stagni)
- I POI commerciali sono proxy imperfetti per la vitalità economica

## ⚙️ Configurazione

Imposta qui i percorsi e i parametri di calcolo.

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURAZIONE - Modifica questi valori se necessario
# ============================================================

# Directory dei file (usa "." per la directory corrente del notebook)
DATA_DIR = Path(".")

# File di input
GEOJSON_FILE = DATA_DIR / "comuni_sardegna.geojson"
CSV_FILE = DATA_DIR / "osm_per_comune.csv"

# File di output
INDICE_CSV = DATA_DIR / "indice_attrattivita.csv"
MAPPA_HTML = DATA_DIR / "mappa_attrattivita_sardegna.html"

# ============================================================
# PESI DEI PILASTRI (somma = 1.0)
# ============================================================
WEIGHTS = {
    "turismo": 0.30,        # 30% - Turismo e patrimonio culturale
    "natura": 0.25,         # 25% - Natura e ambiente
    "ristorazione": 0.20,   # 20% - Ristorazione e commercio
    "servizi": 0.15,        # 15% - Servizi essenziali
    "infrastrutture": 0.10, # 10% - Infrastrutture di trasporto
}

# CRS proiettato per calcolo aree (UTM zone 32N per la Sardegna)
PROJECTED_CRS = "EPSG:32632"

# Coordinate centro mappa (Sardegna)
MAP_CENTER = [40.12, 9.01]
MAP_ZOOM = 8

print(f"✔ Configurazione caricata")
print(f"  DATA_DIR: {DATA_DIR.resolve()}")
print(f"  GEOJSON: {GEOJSON_FILE}")
print(f"  CSV: {CSV_FILE}")
print(f"  Output CSV: {INDICE_CSV}")
print(f"  Output MAP: {MAPPA_HTML}")
print(f"  Somma pesi: {sum(WEIGHTS.values()):.2f}")

## 📦 Installazione Dipendenze

Verifica e installa le librerie necessarie.

In [ ]:
# Librerie core (devono essere installate)
import sys
print("Python version:", sys.version.split()[0])

try:
    import pandas as pd
    print(f"✔ pandas {pd.__version__}")
except ImportError:
    print("✘ pandas non installato: pip install pandas")
    raise

try:
    import geopandas as gpd
    print(f"✔ geopandas {gpd.__version__}")
except ImportError:
    print("✘ geopandas non installato: pip install geopandas")
    raise

# Librerie opzionali per la mappa
try:
    import folium
    print(f"✔ folium {folium.__version__}")
    FOLIUM_AVAILABLE = True
except ImportError:
    print("⚠ folium non installato: pip install folium")
    print("  La generazione della mappa sarà saltata.")
    FOLIUM_AVAILABLE = False

try:
    from branca.element import Element
    print("✔ branca")
except ImportError:
    print("⚠ branca non installato: pip install branca")
    FOLIUM_AVAILABLE = False

## 📂 Caricamento Dati

Carica i confini comunali (GeoJSON) e i conteggi POI (CSV).

In [ ]:
# Verifica esistenza file
if not GEOJSON_FILE.exists():
    raise FileNotFoundError(f"File non trovato: {GEOJSON_FILE}\nEsegui prima il notebook 01")
if not CSV_FILE.exists():
    raise FileNotFoundError(f"File non trovato: {CSV_FILE}\nEsegui prima il notebook 01")

# Carica GeoJSON comuni
print("[1/6] Caricamento dati...")
comuni_gdf = gpd.read_file(GEOJSON_FILE)
print(f"  ✔ {len(comuni_gdf)} comuni caricati")
print(f"  ✔ CRS originale: {comuni_gdf.crs}")

# Carica CSV conteggi POI
counts_df = pd.read_csv(CSV_FILE)
print(f"  ✔ {len(counts_df)} righe nel CSV")
print(f"  ✔ Colonne: {list(counts_df.columns)}")

# Mostra un esempio
print("\n📋 Esempio dati OSM:")
counts_df.head(3)

## 📐 Calcolo Aree Comunali

L'area è calcolata usando un **CRS proiettato (UTM 32N)** per garantire misure metriche accurate. La Sardegna ricade nella zona UTM 32N.

In [ ]:
print("[2/6] Calcolo aree comunali...")

# Riproietta in CRS proiettato per calcolo aree accurate
comuni_projected = comuni_gdf.to_crs(crs=PROJECTED_CRS)

# Calcola area in km² (da m²)
comuni_gdf = comuni_gdf.copy()
comuni_gdf["area_km2"] = comuni_projected.geometry.area / 1_000_000

#Statistiche
print(f"  ✔ Area minima: {comuni_gdf['area_km2'].min():.2f} km²")
print(f"  ✔ Area massima: {comuni_gdf['area_km2'].max():.2f} km²")
print(f"  ✔ Area media: {comuni_gdf['area_km2'].mean():.2f} km²")
print(f"  ✔ Area mediana: {comuni_gdf['area_km2'].median():.2f} km²")

# Verifica valori anomali
too_small = comuni_gdf[comuni_gdf["area_km2"] < 1]
too_large = comuni_gdf[comuni_gdf["area_km2"] > 1000]
print(f"\n  ⚠ Comuni < 1 km²: {len(too_small)}")
print(f"  ⚠ Comuni > 1000 km²: {len(too_large)}")

## 🔢 Normalizzazione per Superficie

I conteggi grezzi sono convertiti in **densità per km²** per rendere comparabili comuni di dimensioni diverse.

La formula è: $D_{pilastro} = \frac{count_{pilastro}}{area\_km2}$

In [ ]:
print("[3/6] Normalizzazione per superficie...")

# Prepara il merge
result_df = counts_df.copy()

# Assicura compatibilità dei codici ISTAT
comuni_areas = comuni_gdf[["com_istat_code", "area_km2"]].copy()
comuni_areas["com_istat_code"] = comuni_areas["com_istat_code"].astype(int)

# Merge per ottenere le aree
result_df = result_df.merge(
    comuni_areas,
    left_on="com_istat_code",
    right_on="com_istat_code",
    how="left"
)

# Mapping colonne: count_xxx -> xxx
column_mapping = {
    "count_turismo": "turismo",
    "count_natura": "natura",
    "count_servizi": "servizi",
    "count_ristorazione": "ristorazione",
    "count_infrastrutture": "infrastrutture"
}
result_df = result_df.rename(columns=column_mapping)

# Calcola densità per ogni pilastro
for pillar in WEIGHTS.keys():
    if pillar in result_df.columns:
        result_df[f"{pillar}_density"] = result_df[pillar] / result_df["area_km2"]
    else:
        result_df[f"{pillar}_density"] = 0
        print(f"  ⚠ Colonna '{pillar}' non trovata, impostata a 0")

print(f"  ✔ Colonne densità create: {[f'{p}_density' for p in WEIGHTS.keys()]}")

# Verifica valori mancanti
missing_area = result_df["area_km2"].isna().sum()
print(f"\n  ⚠ Comuni senza area: {missing_area}")
if missing_area > 0:
    print("  Comuni non appaiati:", result_df[result_df["area_km2"].isna()]["name"].tolist()[:5])

## 📊 Normalizzazione Min-Max (0-100)

Ogni pilastro è scalato in un range 0-100 usando la formula:

$N = \frac{D - D_{min}}{D_{max} - D_{min}} \times 100$

Questo permette di confrontare pilastri con scale diverse.

In [ ]:
print("[4/6] Normalizzazione Min-Max...")

density_cols = [f"{p}_density" for p in WEIGHTS.keys()]

for col in density_cols:
    if col in result_df.columns:
        min_val = result_df[col].min()
        max_val = result_df[col].max()
        
        if max_val > min_val:
            result_df[f"{col.replace('_density', '')}_norm"] = (
                (result_df[col] - min_val) / (max_val - min_val) * 100
            )
        else:
            result_df[f"{col.replace('_density', '')}_norm"] = 0
        
        print(f"  ✔ {col}: min={min_val:.4f}, max={max_val:.4f}")
    else:
        result_df[col.replace("_density", "_norm")] = 0
        print(f"  ⚠ {col} non trovata")

print(f"\n  ✔ Colonne normalizzate: {[f'{p}_norm' for p in WEIGHTS.keys()]}")

## 🧮 Calcolo Indice Ponderato

L'indice finale è la **somma ponderata** dei pilastri normalizzati:

$I = \sum_{pilastro} (N_{pilastro} \times peso_{pilastro})$

In [ ]:
print("[5/6] Calcolo indice ponderato...")

# Calcola l'indice come somma ponderata
result_df["indice"] = 0.0

for pillar, weight in WEIGHTS.items():
    norm_col = f"{pillar}_norm"
    if norm_col in result_df.columns:
        result_df["indice"] += result_df[norm_col] * weight
    print(f"  ✔ {pillar}: peso {weight*100:.0f}%")

print(f"\n  ✔ Indice range: {result_df['indice'].min():.2f} - {result_df['indice'].max():.2f}")
print(f"  ✔ Indice medio: {result_df['indice'].mean():.2f}")
print(f"  ✔ Indice mediano: {result_df['indice'].median():.2f}")

## 🏷️ Classificazione in Quintili

I comuni sono classificati in **5 classi** basate sui quintili dell'indice:

| Classe | Significato |
|--------|-------------|
| 1 | Attrattività molto bassa |
| 2 | Attrattività bassa |
| 3 | Attrattività media |
| 4 | Attrattività alta |
| 5 | Attrattività molto alta |

In [ ]:
print("[6/6] Classificazione in quintili...")

# Classifica in quintili (5 classi)
result_df["classe"] = pd.qcut(
    result_df["indice"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Statistiche per classe
print("\n📊 Distribuzione classi:")
for classe in range(1, 6):
    subset = result_df[result_df["classe"] == classe]
    print(f"  Classe {classe}: {len(subset):3d} comuni "
          f"(indice {subset['indice'].min():.2f} - {subset['indice'].max():.2f})")

## ✅ Controlli di Qualità

Verifica la qualità dei dati: righe, valori mancanti, range, distribuzione.

In [ ]:
print("=" * 60)
print("CONTROLLI DI QUALITÀ")
print("=" * 60)

# 1. Conteggio righe
print(f"\n1. Conteggio righe:")
print(f"   Comuni nel GeoJSON: {len(comuni_gdf)}")
print(f"   Comuni nel CSV: {len(counts_df)}")
print(f"   Comuni con indice: {len(result_df)}")

# 2. Valori mancanti
print(f"\n2. Valori mancanti per pilastro:")
for pillar in WEIGHTS.keys():
    missing = result_df[pillar].isna().sum()
    zero = (result_df[pillar] == 0).sum()
    print(f"   {pillar}: {missing} NaN, {zero} zeri")

# 3. Range degli score
print(f"\n3. Range degli score normalizzati (0-100):")
for pillar in WEIGHTS.keys():
    norm_col = f"{pillar}_norm"
    if norm_col in result_df.columns:
        print(f"   {pillar}: {result_df[norm_col].min():.1f} - {result_df[norm_col].max():.1f}")

# 4. Distribuzione classi
print(f"\n4. Distribuzione classi:")
class_dist = result_df["classe"].value_counts().sort_index()
for classe, count in class_dist.items():
    pct = count / len(result_df) * 100
    print(f"   Classe {classe}: {count:3d} comuni ({pct:.1f}%)")

# 5. TOP 10 comuni
print(f"\n5. TOP 10 COMUNI (migliori):")
top10 = result_df.nlargest(10, "indice")[["name", "indice", "classe"]]
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"   {i:2}. {row['name']:<25} indice={row['indice']:>6.2f} classe={row['classe']}")

# 6. BOTTOM 10 comuni
print(f"\n6. BOTTOM 10 COMUNI (peggiori):")
bottom10 = result_df.nsmallest(10, "indice")[["name", "indice", "classe"]]
for i, (_, row) in enumerate(bottom10.iterrows(), 1):
    print(f"   {i:2}. {row['name']:<25} indice={row['indice']:>6.2f} classe={row['classe']}")

## 💾 Esportazione CSV

Salva il risultato in `indice_attrattivita.csv`.

In [ ]:
# Seleziona le colonne rilevanti
output_df = result_df[[
    "com_istat_code",
    "name",
    "area_km2",
    "turismo", "natura", "ristorazione", "servizi", "infrastrutture",
    "indice",
    "classe"
]].copy()

# Rinomina per chiarezza
output_df = output_df.rename(columns={
    "com_istat_code": "codice_istat",
    "name": "nome_comune"
})

# Arrotonda i valori numerici
output_df["area_km2"] = output_df["area_km2"].round(2)
output_df["indice"] = output_df["indice"].round(2)

# Salva
output_df.to_csv(INDICE_CSV, index=False, encoding="utf-8")
print(f"✔ CSV salvato: {INDICE_CSV}")
print(f"  ✔ {len(output_df)} righe")

# Anteprima
output_df.head(5)

## 🗺️ Generazione Mappa Interattiva

Genera una mappa coropletica HTML con Folium. La mappa mostra l'indice di attrattività per comune con tooltip interattivi.

In [ ]:
if not FOLIUM_AVAILABLE:
    print("⚠ FOLIUM NON È INSTALLATO")
    print("   Per installare: pip install folium branca")
    print("   La mappa non sarà generata.")
else:
    from folium import MacroElement
    from branca.element import Element
    
    print("Generazione mappa coropletica...")
    
    # Prepara i dati per la mappa
    comuni_for_map = comuni_gdf.merge(
        result_df[["com_istat_code", "indice", "classe"] + 
                  [f"{p}_norm" for p in WEIGHTS.keys()]],
        left_on="com_istat_code",
        right_on="com_istat_code",
        how="left"
    )
    
    # Crea la mappa base
    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles="CartoDB positron"
    )
    
    # Costruisci il GeoJSON inline
    geojson_data = comuni_for_map.to_json()
    
    # Crea la mappa coropletica
    folium.Choropleth(
        geo_data=geojson_data,
        name="Attrattività",
        data=comuni_for_map,
        columns=["com_istat_code", "indice"],
        key_on="feature.properties.com_istat_code",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.2,
        line_weight=1,
        legend_name="Indice di Attrattività",
        nan_fill_color="white",
        nan_fill_opacity=0.3
    ).add_to(m)
    
    # Tooltip interattivo
    style_function = lambda x: {
        "fillColor": "#ffffff",
        "color": "#000000",
        "fillOpacity": 0,
        "weight": 0
    }
    
    highlight_function = lambda x: {
        "fillColor": "#000000",
        "color": "#000000",
        "fillOpacity": 0.1,
        "weight": 1
    }
    
    # Crea il layer GeoJson con tooltip
    folium.GeoJson(
        geojson_data,
        name="Dettagli comune",
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "indice", "classe",
                    "turismo_norm", "natura_norm", "ristorazione_norm",
                    "servizi_norm", "infrastrutture_norm"],
            aliases=["Comune:", "Indice:", "Classe (1-5):",
                    "Turismo:", "Natura:", "Ristorazione:",
                    "Servizi:", "Infrastrutture:"],
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: white;
                border: 2px solid black;
                border-radius: 3px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);
                font-size: 12px;
                padding: 10px;
            """
        )
    ).add_to(m)
    
    # Pannello info HTML custom
    info_html = """
    <div style="position: fixed; 
                top: 10px; right: 10px; 
                width: 220px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 12px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0; border-bottom: 1px solid grey; padding-bottom: 5px;">
            Indice di Attrattività
        </h4>
        <p style="margin: 5px 0;">
            <b>5 pilastri tematici</b><br>
            Turismo, Natura, Ristorazione,<br>
            Servizi, Infrastrutture
        </p>
        <p style="margin: 5px 0;">
            <b>Classi di attrattività:</b><br>
            1 = Molto bassa<br>
            2 = Bassa<br>
            3 = Media<br>
            4 = Alta<br>
            5 = Molto alta
        </p>
        <p style="margin: 5px 0; font-size: 10px; color: #666;">
            Passa il mouse sui comuni per i dettagli
        </p>
    </div>
    """
    m.get_root().html.add_child(Element(info_html))
    
    # Legenda custom
    legend_html = """
    <div style="position: fixed; 
                bottom: 50px; left: 10px; 
                width: 150px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 11px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0;">Legenda</h4>
        <div style="background: linear-gradient(to right, #FFFFB2, #FED976, #FEB24C, #FD8D3C, #FC4E2A, #E31A1C, #B10026); 
                    height: 20px; 
                    border-radius: 3px;
                    margin-bottom: 5px;"></div>
        <div style="display: flex; justify-content: space-between; font-size: 10px;">
            <span>Basso</span>
            <span>Alto</span>
        </div>
    </div>
    """
    m.get_root().html.add_child(Element(legend_html))
    
    # Controllo layer
    folium.LayerControl().add_to(m)
    
    # Salva la mappa
    m.save(MAPPA_HTML)
    print(f"✔ Mappa salvata: {MAPPA_HTML}")

## 📈 Visualizzazione Top/Bottom 10

Grafico a barre dei 10 comuni migliori e peggiori per indice di attrattività.

In [ ]:
import matplotlib.pyplot as plt

# TOP 10
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top10 = result_df.nlargest(10, "indice")[["name", "indice", "classe"]].sort_values("indice")
axes[0].barh(top10["name"], top10["indice"], color="#2ecc71")
axes[0].set_xlabel("Indice di Attrattività")
axes[0].set_title("🏆 TOP 10 - Comuni Migliori", fontsize=12, fontweight="bold")
axes[0].set_xlim(0, 100)
for i, v in enumerate(top10["indice"]):
    axes[0].text(v + 1, i, f"{v:.1f}", va="center", fontsize=9)

# BOTTOM 10
bottom10 = result_df.nsmallest(10, "indice")[["name", "indice", "classe"]].sort_values("indice")
axes[1].barh(bottom10["name"], bottom10["indice"], color="#e74c3c")
axes[1].set_xlabel("Indice di Attrattività")
axes[1].set_title("📉 BOTTOM 10 - Comuni Peggiori", fontsize=12, fontweight="bold")
axes[1].set_xlim(0, 100)
for i, v in enumerate(bottom10["indice"]):
    axes[1].text(v + 1, i, f"{v:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / "top_bottom_10.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✔ Grafico salvato: {DATA_DIR / 'top_bottom_10.png'}")

---

## ✅ Riepilogo

**File generati:**
- `indice_attrattivita.csv`: dati completi con indice e classe
- `mappa_attrattivita_sardegna.html`: mappa interattiva (se folium installato)
- `top_bottom_10.png`: grafico top/bottom 10

**Prossimi passi:**
- Esegui il notebook 03 per calcolare l'indice di overtourism
- Consulta la mappa HTML per un'analisi visiva spaziale